# MNIST MLP 분류기

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from typing import List

## 1. 데이터셋

In [ ]:
# ToTensor는 클래스라서 ToTensor() 로 인스턴스를 만들어 넘겨야 함
# 테스트셋은 test=True 가 아니라 train=False
mnist_train = datasets.MNIST(root="./mnist_data", train=True, download=True, transform=transforms.ToTensor())
mnist_test = datasets.MNIST(root="./mnist_data", train=False, download=True, transform=transforms.ToTensor())

In [ ]:
print(len(mnist_train), len(mnist_test))
print(mnist_train[0][0].shape, mnist_train[0][1])

## 2. DataLoader

In [ ]:
batch_size = 32
train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=False)

In [ ]:
train_iter = iter(train_loader)
images, labels = next(train_iter)
print(images.shape, labels.shape)  # torch.Size([32, 1, 28, 28]) torch.Size([32])

## 3. 모델

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_layer_dims: List[int], output_dim: int):
        super(MLPClassifier, self).__init__()
        self.input_dim = input_dim
        self.hidden_layers = nn.ModuleList()

        prev_dim = input_dim
        for layer_dim in hidden_layer_dims:
            hidden_layer = nn.Linear(prev_dim, layer_dim, bias=True)
            self.hidden_layers.append(hidden_layer)
            prev_dim = layer_dim

        self.output_layer = nn.Linear(prev_dim, output_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (B, 1, 28, 28) -> (B, 784): 배치 차원은 유지하고 나머지만 펼침
        x = x.view(-1, self.input_dim)
        for layer in self.hidden_layers:
            x = F.relu(layer(x))
        # softmax는 CrossEntropyLoss 안에 포함돼 있으므로 logits 그대로 반환
        x = self.output_layer(x)
        return x

## 4. 하이퍼파라미터 · 손실함수 · 옵티마이저

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

num_epochs = 5
learning_rate = 0.01

input_dimension = 28 * 28
output_dimension = 10
hidden_layer_sizes = [128, 128, 128]

model = MLPClassifier(input_dimension, hidden_layer_sizes, output_dimension)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.5)

print(model)

## 5. 학습 / 평가 함수

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total

## 6. 학습 루프

In [ ]:
for epoch in range(1, num_epochs + 1):
    current_lr = lr_scheduler.get_last_lr()[0]
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    lr_scheduler.step()

    print(f"[Epoch {epoch}/{num_epochs}] lr={current_lr:.5f} | "
          f"train loss {train_loss:.4f}, acc {train_acc*100:.2f}% | "
          f"test loss {test_loss:.4f}, acc {test_acc*100:.2f}%")

## 7. 예측 확인

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(images.to(device)).argmax(dim=1).cpu()

print("정답:", labels[:10].tolist())
print("예측:", preds[:10].tolist())